# Analyze Energy Ratios during Baseline Operation

In this notebook, we will demonstrate how to compute and plot the energy ratio between test and reference turbines as a function of wind direction. We'll focus on baseline operation for this example (i.e., without wake steering). The energy ratios can be used to evaluate wake losses experienced by different turbines.

In [18]:
from pathlib import Path

import numpy as np
import pandas as pd

from flasc.analysis import energy_ratio as er
from flasc.analysis.analysis_input import AnalysisInput
from flasc.data_processing import dataframe_manipulations as dfm
from flasc.utilities import floris_tools as ftools
from flasc.utilities.utilities_examples import load_floris_smarteole as load_floris

In [19]:
# Suppress warnings
import warnings

warnings.filterwarnings("ignore")

# Step 0: Load processed data

Load the processed SCADA data with power curve filtering and northing calibration applied and inspect

In [20]:
def load_data():
    root_path = Path.cwd()
    f = root_path / "postprocessed" / "df_scada_data_60s_filtered_and_northing_calibrated.pkl"
    df_scada = pd.read_pickle(f)

    return df_scada


df_scada = load_data()

In [21]:
df_scada.describe()

,time,pow_000,pow_001,pow_002,pow_003,pow_004,pow_005,pow_006,ws_000,ws_001,...,wd_004,wd_005,wd_006,wind_vane_005,target_yaw_offset_005,wd_smarteole,ws_smarteole,pow_ref_smarteole,ti,wd
count,134661,107963.000000,104981.000000,91702.000000,100774.000000,105342.000000,99677.000000,106162.000000,107963.000000,104981.000000,...,105342.000000,99677.000000,106162.000000,99677.000000,99677.000000,131704.000000,131699.000000,130266.000000,1.346610e+05,115725.000000
mean,2020-04-04 18:49:02.648873472,777.708679,722.314758,788.448669,650.102112,714.342102,707.234131,734.999878,7.992576,7.441976,...,147.802322,152.649872,145.970215,0.825125,0.786913,147.975891,7.286752,639.086670,1.100000e-01,147.127971
min,2020-02-17 16:30:00,0.001000,0.003000,0.002000,0.009000,0.007000,0.032000,0.000000,1.483000,2.746000,...,0.022003,0.009750,0.191750,-43.056999,-0.000000,0.005929,0.101000,-45.498859,1.100000e-01,0.035299
25%,2020-03-12 01:51:00,235.326004,214.001999,246.753754,192.749237,205.838989,173.522003,219.312500,6.128000,5.577000,...,50.160999,52.731747,49.358002,-5.912000,0.000000,47.813652,5.121595,98.970131,1.100000e-01,47.609299
50%,2020-04-04 10:58:00,571.359985,528.625000,593.072998,467.079498,504.357483,504.005005,540.981506,7.448000,7.027000,...,126.497002,163.868759,134.247757,0.187000,0.000000,138.522034,7.423318,422.463989,1.100000e-01,136.630460
75%,2020-04-27 20:03:00,1238.735962,1119.993042,1244.010986,997.394531,1098.766724,1124.659058,1134.179199,9.587500,8.942000,...,235.623001,235.056763,232.824249,6.927000,0.000000,235.295410,9.361652,1049.996338,1.100000e-01,233.829382
max,2020-05-24 23:59:00,2064.696045,2065.387939,2051.489990,2062.987061,2064.689941,2078.750977,2076.895996,20.851999,21.240999,...,359.997009,359.980743,359.988770,59.181999,20.000000,359.991791,20.422459,2147.802246,1.100000e-01,359.957790
std,NaN,639.332153,609.761963,636.423706,560.708984,617.146301,622.352234,618.290527,2.548205,2.501244,...,99.606598,97.270058,97.408218,10.048730,3.216291,101.749588,3.104111,629.421570,1.387784e-17,100.096957


In [22]:
df_scada.columns

Index(['time', 'pow_000', 'pow_001', 'pow_002', 'pow_003', 'pow_004',
       'pow_005', 'pow_006', 'ws_000', 'ws_001', 'ws_002', 'ws_003', 'ws_004',
       'ws_005', 'ws_006', 'wd_000', 'wd_001', 'wd_002', 'wd_003', 'wd_004',
       'wd_005', 'wd_006', 'is_operation_normal_000',
       'is_operation_normal_001', 'is_operation_normal_002',
       'is_operation_normal_003', 'is_operation_normal_004',
       'is_operation_normal_005', 'is_operation_normal_006', 'wind_vane_005',
       'target_yaw_offset_005', 'control_mode', 'wd_smarteole', 'ws_smarteole',
       'pow_ref_smarteole', 'ti', 'wd'],
      dtype='object')

# Step 1: Limit to baseline data

The SCADA data set contains alternating 1-hour periods with baseline or wake steering control. For these examples, we'll limit the data to baseline operation. 

In [23]:
df_scada = df_scada[df_scada.control_mode == "baseline"]

# Step 2: Compute reference wind direction, wind speed, and power variables

The energy ratio class as presently implemented requires explicit identification of the dataframe of columns "wd," "ws," and "pow_ref." We'll use the FLORIS model to establish which turbines are unwaked for each wind direction to compute the reference variables.

In [24]:
# Load FLORIS model of site
fm, turbine_weights = load_floris()

In [25]:
# Use FLORIS to identify upstream / unwaked turbines for
# each direction
df_upstream = ftools.get_upstream_turbs_floris(fm)

df_upstream.head()

,wd_min,wd_max,turbines
0,0.0,25.0,[0]
1,25.0,28.0,"[0, 2]"
2,28.0,31.0,"[0, 2, 6]"
3,31.0,31.3,"[0, 2, 3, 6]"
4,31.3,31.5,"[0, 2, 3, 4, 6]"


In [26]:
# Use flasc tools to establish reference wind speeds and directions

# Since will be interested in looking at impacts on SMV5/[4], exclude
# it from each calculation

# Set the wind direction as the average of all turbine averages
df_scada = dfm.set_wd_by_turbines(df_scada, [0, 1, 2, 3, 5, 6])

# Set the wind speed to be the average of all upstream turbines
# (turbines not in a wake in a given direction)
# Except for SMV5
df_scada = dfm.set_ws_by_upstream_turbines(df_scada, df_upstream, exclude_turbs=[4])

# Set the reference power to the average of all upstream turbines
# Except for SMV5
df_scada = dfm.set_pow_ref_by_upstream_turbines(df_scada, df_upstream, exclude_turbs=[4])

# Step 3: Compute and Plot Energy Ratio for Turbine 004

Compare the energy ratio for turbine 004 based on the SCADA data to the equivalent predicted energy ratios from each FLORIS model using our precomputed FLORIS results. Turbine 004 is the downstream waked turbine that benefits from wake steering in the SMARTEOLE wake steering field experiment.

The energy ratios computed here simply represent the ratio between the energy produced by the test turbines and the energy computed using the reference power variable, "pow_ref," as a function of wind direction.

In [48]:
# Grab the LES timeseries
df_les_timeseries_raw = pd.read_csv("data/SMARTEOLE-LES-simulation-data/les_timeseries.csv")
df_les_timeseries_raw["time"] = pd.to_datetime(df_les_timeseries_raw["time"])
print(df_les_timeseries_raw)

                      time  wd_metmast  ws_metmast  ti_metmast       pow_000  \
0      2020-02-17 16:30:00  251.703429   11.846795    0.065286  1.957574e+06   
1      2020-02-17 16:31:00  251.163184   13.082008    0.065286  1.970251e+06   
2      2020-02-17 16:32:00  253.319797   10.564166    0.065286  1.987973e+06   
3      2020-02-17 16:33:00  251.491886   11.302930    0.065286  2.017250e+06   
4      2020-02-17 16:34:00  255.584759   10.600017    0.065286  1.999241e+06   
...                    ...         ...         ...         ...           ...   
140125 2020-05-24 23:55:00  310.416653    4.140377    0.065286  2.007438e+05   
140126 2020-05-24 23:56:00  307.748918    3.932864    0.065286  1.995100e+05   
140127 2020-05-24 23:57:00  311.550865    4.706409    0.065286  1.973253e+05   
140128 2020-05-24 23:58:00  306.302918    4.387220    0.065286  1.906684e+05   
140129 2020-05-24 23:59:00  310.633269    5.162499    0.065286  2.035454e+05   

             pow_001       pow_002     

In [87]:
# Option for validation 1: compare the two timeseries directly, assuming (hoping) that LES predicted the inflow conditions accurately
# For this, we must ensure the NaNs in the SCADA also appear as NaNs in the LES timeseries. Assuming the same timezone format...

# First map LES to the SCADA timeseries using linear interpolation
from flasc.data_processing import time_operations as tops
df_les = tops.df_resample_by_interpolation(
    df=df_les_timeseries_raw,
    time_array=pd.to_datetime(df_scada["time"]),
    circular_cols=["wd_metmast"],
)

# Now being on the same timeseries, map NaNs
n_turbs = dfm.get_num_turbines(df_scada)
for ti in range(n_turbs):
    ids_nan = np.where(df_scada[f"pow_{ti:03d}"].isna())[0]
    n = len(ids_nan)
    print(f"Copying over {n} NaNs from SCADA to LES timeseries for 'pow_{ti:03d}'...")
    df_les.loc[ids_nan, f"pow_{ti:03d}"] = None

pow_cols = [f"pow_{ti:03d}" for ti in range(n_turbs)]
# n_scada = df_scada[pow_cols].isna().sum().sum()
# n_les = df_les[pow_cols].isna().sum().sum()
# print(f"NaNs in df_scada power columns: {n_scada}")
# print(f"NaNs in df_les power columns: {n_les}")

# Fix scaling in LES from [W] to [kW]
df_les[pow_cols] *= 1.0e-3

# Compare AEP directly
unit_conversion = 1.0e-3 / (np.timedelta64(3600, 's') / np.nanmedian(df_scada["time"].diff()))  # From sum of samples to MWh
aep_scada = df_scada[pow_cols].sum().sum() * unit_conversion  # In MWh
aep_les = df_les[pow_cols].sum().sum() * unit_conversion  # In MWh
diff = 100.0 * (aep_les - aep_scada) / aep_scada
print(f"Absolute AEP.  SCADA: {aep_scada:.1f} MWh. LES: {aep_les:.1f} MWh. Diff: {diff:+.1f} %.")

# Compare AEP wake loss relative to most upstream turbines
df_les["wd"] = df_les["wd_metmast"]
df_les = dfm.set_pow_ref_by_upstream_turbines(df_les, df_upstream, exclude_turbs=[4])  # Set reference power
aep_scada_unwaked = np.sum(df_scada["pow_ref"] * n_turbs) * unit_conversion  # In MWh
aep_les_unwaked = np.sum(df_les["pow_ref"] * n_turbs) * unit_conversion  # In MWh

wake_loss_scada = 100.0 * (aep_scada_unwaked - aep_scada) / aep_scada_unwaked
wake_loss_les = 100.0 * (aep_les_unwaked - aep_les) / aep_les_unwaked
diff = wake_loss_les - wake_loss_scada
print(f"AEP wake loss. SCADA: {wake_loss_scada:.2f} %. LES: {wake_loss_les:.2f} %. Diff: {diff:+.2f} %-point.")

2025-11-04 16:55:47   Resampling column 'wd_metmast' with median timestep 60.000 s onto a prespecified time array with kind=linear, max_gap=90.0s, and wrap_around_360=True
2025-11-04 16:55:47   Resampling column 'ws_metmast' with median timestep 60.000 s onto a prespecified time array with kind=linear, max_gap=90.0s, and wrap_around_360=False
2025-11-04 16:55:48   Resampling column 'ti_metmast' with median timestep 60.000 s onto a prespecified time array with kind=linear, max_gap=90.0s, and wrap_around_360=False
2025-11-04 16:55:48   Resampling column 'pow_000' with median timestep 60.000 s onto a prespecified time array with kind=linear, max_gap=90.0s, and wrap_around_360=False
2025-11-04 16:55:48   Resampling column 'pow_001' with median timestep 60.000 s onto a prespecified time array with kind=linear, max_gap=90.0s, and wrap_around_360=False
2025-11-04 16:55:48   Resampling column 'pow_002' with median timestep 60.000 s onto a prespecified time array with kind=linear, max_gap=90.0s

Copying over 22673 NaNs from SCADA to LES timeseries for 'pow_000'...
Copying over 24570 NaNs from SCADA to LES timeseries for 'pow_001'...
Copying over 32284 NaNs from SCADA to LES timeseries for 'pow_002'...
Copying over 27352 NaNs from SCADA to LES timeseries for 'pow_003'...
Copying over 24925 NaNs from SCADA to LES timeseries for 'pow_004'...
Copying over 30129 NaNs from SCADA to LES timeseries for 'pow_005'...
Copying over 25127 NaNs from SCADA to LES timeseries for 'pow_006'...
Absolute AEP.  SCADA: 4520.7 MWh. LES: 4446.2 MWh. Diff: -1.6 %.
AEP wake loss. SCADA: 13.94 %. LES: 15.08 %. Diff: +1.14 %-point.


In [ ]:
# Option for validation 2: map the LES solutions to the SCADA inflow timeseries by turning the LES simulation into a steady-state solution table
# Get a list of precalculated FLORIS results
floris_path = Path.cwd() / "precalculated_floris_solutions"
wake_models = ["jensen", "gch", "cc", "turbopark"]
df_fm_list = [None for _ in wake_models]
for wii, wake_model in enumerate(wake_models):
    fn = floris_path / "df_fi_approx_{:s}.ftr".format(wake_model)
    if fn.is_file():
        df_fm_approx = pd.read_feather(fn)
    else:
        raise UserWarning(
            "Please run '01_precalculate_floris_solutions.ipynb' "
            "for the appropriate wake models first."
        )

    df_fm_list[wii] = ftools.interpolate_floris_from_df_approx(
        df=df_scada, df_approx=df_fm_approx, method="linear", verbose=True
    )

In [28]:
# df_scada["time"].diff().unique()
# from flasc.data_processing import time_operations as tops
# new_time_array = pd.date_range(start=df_scada.iloc[0]["time"], end=df_scada.iloc[-1]["time"], freq="600s")
# print(new_time_array)
# df_res = tops.df_resample_by_interpolation(
#     df=df_scada[["time", "wd", "ws", "ti"]],
#     time_array=new_time_array,
#     circular_cols=["wd"],
#     interp_method="linear",
#     verbose=True,
# )
# print(df_res)


In [29]:
# # Set pow_ref in FLORIS results as before
# for df_fm in df_fm_list:
#     df_fm = dfm.set_pow_ref_by_upstream_turbines(df_fm, df_upstream, exclude_turbs=[4])

In [30]:
# # Calculate and plot energy ratios
# a_in = AnalysisInput(
#     df_fm_list + [df_scada], ["FLORIS: " + wm for wm in wake_models] + ["SCADA data"]
# )

In [31]:
# N = 20
# print("Calculating energy ratios with bootstrapping (N={}).".format(N))
# print("This may take a couple seconds...")
# np.random.seed(0)
# er_out = er.compute_energy_ratio(
#     a_in,
#     test_turbines=[4],
#     use_predefined_ref=True,
#     use_predefined_wd=True,
#     use_predefined_ws=True,
#     wd_step=2.0,
#     wd_bin_overlap_radius=0.0,
#     ws_min=6.0,
#     ws_max=12.0,
#     N=N,
#     percentiles=[5.0, 95.0],
# )
# ax = er_out.plot_energy_ratios(overlay_frequency=True)
# ax[0].set_title("Energy Ratios for Turbine 004")

As shown in the plot above, overall there is good agreement between the SCADA-based energy ratio curve and the energy ratio predictions based on FLORIS results. However, because of relatively little data for many wind directions, the SCADA-based energy ratios are noisy and can deviate from the expected value of 1 when Turbine 004 is unwaked.

# Step 4: Rerun the energy ratio calculation with a different wind speed/wind direction distribution

As an example, we'll create a distribution with uniform frequency across 
all wind speds of interest and concentrated in the direction where SMV6 wakes SMV5.

Can also be used to evaluate the energy ratios under long-term site conditions.

In [32]:
# ws = np.tile(np.arange(6.5, 12.0, 1.0), 180)
# wd = np.repeat(np.arange(1.0, 360.0, 2.0), 6)

# freq = np.ones_like(ws)
# # Increase frequency value in steering wind directions
# start_idx = np.where(wd == 169)[0][0]
# end_idx = np.where(wd == 231)[0][0]
# freq[start_idx:end_idx] = 5
# freq = 10 * freq

# df_freq = pd.DataFrame({"ws": ws, "wd": wd, "freq_val": freq})

# N = 20
# print("Calculating energy ratios with bootstrapping (N={}).".format(N))
# print("This may take a couple seconds...")
# np.random.seed(0)
# er_out = er.compute_energy_ratio(
#     a_in,
#     test_turbines=[4],
#     use_predefined_ref=True,
#     use_predefined_wd=True,
#     use_predefined_ws=True,
#     wd_step=2.0,
#     wd_bin_overlap_radius=0.0,
#     ws_min=6.0,
#     ws_max=12.0,
#     df_freq=df_freq,
#     N=N,
#     percentiles=[5.0, 95.0],
# )
# ax = er_out.plot_energy_ratios(overlay_frequency=True)
# ax[0].set_title("Energy Ratios for Turbine 004")